# Stored execution evidence — Dog Breed Identification

This public evidence copy preserves every textual output cell supplied in `mle.zip`.
The original notebook SHA-256 is `fbc0aed882b76adacfb2d917c01a7df460f687a329dd829f1429738c68e57a37`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Standalone MLE-STAR benchmark runner

This is the original compact workflow, extended without changing its seven-dataset
order: each dataset still has one visible execution cell. Shared implementation is
kept in four collapsed helper cells.

For every dataset the cell records OOF MLE-STAR results, reconstructs the selected
ensemble, predicts the Kaggle test set, validates against `sample_submission.csv`,
and optionally submits through the Kaggle API. Vision epochs are selected on an
inner validation split; `MAX_EPOCHS` is only a ceiling.


In [1]:
# Reproducible repository setup for Colab.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY = "https://github.com/Isso-W/Jiaozi.git"
BRANCH = "main"
REPOSITORY_ROOT = Path("/content/Jiaozi")
EXPERIMENT_ROOT = REPOSITORY_ROOT / "experiments" / "mlestar_kaggle_benchmarks"

if REPOSITORY_ROOT.exists():
    shutil.rmtree(REPOSITORY_ROOT)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(REPOSITORY_ROOT)],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(REPOSITORY_ROOT), "rev-parse", "HEAD"], text=True
).strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"{EXPERIMENT_ROOT}[vision,llm,kaggle,dev]"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle==2.2.2"],
    check=True,
)
os.chdir(EXPERIMENT_ROOT)
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))
print("Repository commit:", actual_commit)
subprocess.run([sys.executable, "-m", "mlestar.cli", "benchmarks"], check=True)


Repository commit: 01228290ba427023f3e3d386255558f50b21a1a6


CompletedProcess(args=['/usr/bin/python3', '-m', 'mlestar.cli', 'benchmarks'], returncode=0)

In [2]:
# Credentials and experiment controls.
# Add KAGGLE_API_TOKEN in Colab: left sidebar -> key icon -> Secrets.
try:
    from google.colab import userdata
except ImportError:
    userdata = None

def load_colab_secret(name):
    if userdata is None:
        return
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value

load_colab_secret("KAGGLE_API_TOKEN")
load_colab_secret("HF_TOKEN")
if os.environ.get("HF_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
os.environ["HF_HOME"] = "/content/hf_cache"

# Run All executes these seven pipelines from top to bottom. Comment out a
# name only when intentionally resuming or debugging one task.
BENCHMARKS_TO_RUN = {
    "leaf_classification",
    "plant_pathology_2020",
    "aptos_2019",
    "dog_breed",
    "aerial_cactus",
    "dogs_vs_cats",
    "histopathologic_cancer",
}

SEEDS = (13, 29, 47)
IMAGE_SIZE = 128
# Epoch count is selected inside each training run from an inner validation
# split. MAX_EPOCHS is only a safety ceiling, not the chosen epoch count.
MAX_EPOCHS = 40
MIN_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 5
INNER_VALIDATION_FRACTION = 0.10
BATCH_SIZE = 32
MAX_TRAIN_SAMPLES_PER_FOLD = 2000
NUM_WORKERS = 4
USE_AMP = True  # Set False for strict FP32; AMP can cause tiny numerical differences.

# Every task submits only after its sample-submission schema and IDs pass.
SUBMIT_ALL_VIA_API = True
SUBMISSION_PREFIX = "MLE-STAR unified benchmark"
KAGGLE_SCORE_POLL_SECONDS = 300
# Make genuine data/training/prediction failures visibly red in Colab.
# Kaggle submission rejection is captured as a result rather than raised.
STOP_ON_PIPELINE_FAILURE = True

# Persist OOF artifacts, selected epochs, final checkpoints and submissions.
# Data archives remain under /content and may need to be downloaded again.
PERSIST_RUNS_TO_GOOGLE_DRIVE = True
RESUME_EXISTING_RUNS = True
# Allows recovery of the older notebook's completed OOF artifacts, which
# used fixed 3 epochs. Such a submission is labelled legacy and must not be
# mixed with the new inner-selected-epoch OOF comparison.
ALLOW_LEGACY_FIXED_EPOCH_OOF_RESUME = False

LEGACY_RUNS_ROOT = Path("/content/mlestar-runs")
if PERSIST_RUNS_TO_GOOGLE_DRIVE and userdata is not None:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_ROOT = Path("/content/drive/MyDrive/mlestar-runs")
    # Rescue artifacts made by the older notebook in this still-live VM.
    # If the VM was already reset, /content has no old files to recover.
    if LEGACY_RUNS_ROOT.is_dir():
        shutil.copytree(LEGACY_RUNS_ROOT, RUNS_ROOT, dirs_exist_ok=True)
else:
    RUNS_ROOT = LEGACY_RUNS_ROOT
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print("Kaggle token configured:", bool(os.environ.get("KAGGLE_API_TOKEN")))
print("Selected benchmarks:", sorted(BENCHMARKS_TO_RUN))
print("Persistent runs root:", RUNS_ROOT)


Mounted at /content/drive
Kaggle token configured: True
Selected benchmarks: ['aerial_cactus', 'aptos_2019', 'dog_breed', 'dogs_vs_cats', 'histopathologic_cancer', 'leaf_classification', 'plant_pathology_2020']
Persistent runs root: /content/drive/MyDrive/mlestar-runs


In [3]:
# Reusable Kaggle competition fetcher from the original notebook, rewritten
# without notebook shell interpolation so failures retain stdout/stderr.
import zipfile

def fetch_kaggle_competition(slug: str, data_root: str | Path, marker_file: str) -> Path:
    root = Path(data_root)
    root.mkdir(parents=True, exist_ok=True)
    if (root / marker_file).exists():
        print("Dataset already present:", root)
        return root
    if not os.environ.get("KAGGLE_API_TOKEN"):
        raise RuntimeError(
            f"No {marker_file} in {root} and KAGGLE_API_TOKEN is not configured."
        )
    command = [
        "kaggle", "competitions", "download", "-c", slug,
        "-p", str(root),
    ]
    completed = subprocess.run(
        command, text=True, capture_output=True, check=False,
    )
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.returncode != 0:
        detail = completed.stderr.strip() or completed.stdout.strip() or "No error text returned"
        if "403" in detail or "Forbidden" in detail:
            raise PermissionError(
                f"Kaggle denied access to {slug}. The CLI reached Kaggle, but the "
                "account represented by KAGGLE_API_TOKEN cannot download this "
                "competition's files. Open that competition's Rules page while "
                "signed into the same account, join/accept it, then generate a new "
                f"token and retry. Kaggle response:\n{detail}"
            )
        raise RuntimeError(
            f"Kaggle download failed for {slug} (exit {completed.returncode}).\n"
            f"Command: {' '.join(command)}\n"
            f"Kaggle response:\n{detail}"
        )
    outer_zip = root / f"{slug}.zip"
    if outer_zip.exists():
        with zipfile.ZipFile(outer_zip) as archive:
            archive.extractall(root)
    for nested_zip in root.glob("*.zip"):
        if nested_zip == outer_zip:
            continue
        with zipfile.ZipFile(nested_zip) as archive:
            archive.extractall(root)
    if not (root / marker_file).exists():
        raise RuntimeError(
            f"Download did not produce {marker_file} in {root}. Inspect the Kaggle CLI "
            "output and confirm that the token is current and rules are accepted."
        )
    return root


In [4]:
#@title APTOS special preprocessing (shared helper) { display-mode: "form" }
# APTOS-only lossless resize cache. It applies exactly the original adapter's
# Image.open(...).convert("RGB").resize((128, 128)) operation once, then stores
# the result as PNG. Ten images are checked for exact pixel equality.
import json
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

def prepare_aptos_cache(source_root: str | Path) -> Path:
    source_root = Path(source_root)
    cache_root = Path("/content/aptos2019-resized-128")
    cache_root.mkdir(parents=True, exist_ok=True)
    for csv_name in ("train.csv", "test.csv", "sample_submission.csv"):
        source = source_root / csv_name
        if source.is_file():
            shutil.copy2(source, cache_root / csv_name)

    work = []
    for split in ("train_images", "test_images"):
        source_dir = source_root / split
        destination_dir = cache_root / split
        destination_dir.mkdir(parents=True, exist_ok=True)
        work.extend(
            (path, destination_dir / path.name)
            for path in sorted(source_dir.glob("*.png"))
        )
    if not work:
        raise RuntimeError(f"No APTOS PNG images found in {source_root}")

    def resize_one(item):
        source, destination = item
        if destination.is_file():
            return False
        with Image.open(source) as opened:
            resized = opened.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
            temporary = destination.with_suffix(".tmp.png")
            resized.save(temporary, format="PNG", compress_level=1)
            temporary.replace(destination)
        return True

    created = 0
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        for result in tqdm(
            executor.map(resize_one, work), total=len(work), desc="APTOS lossless cache"
        ):
            created += int(result)
    missing = [str(destination) for _, destination in work if not destination.is_file()]
    if missing:
        raise RuntimeError(f"Incomplete APTOS cache, examples: {missing[:5]}")

    for source, cached in work[:10]:
        with Image.open(source) as opened:
            expected = np.asarray(
                opened.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE)), dtype=np.uint8
            )
        with Image.open(cached) as opened:
            actual = np.asarray(opened.convert("RGB"), dtype=np.uint8)
        if not np.array_equal(expected, actual):
            raise AssertionError(f"Cached pixels differ for {source.name}")
    print({"cached_images": len(work), "new_files": created, "pixel_checks": 10})
    return cache_root


In [5]:
#@title MLE-selected epoch vision adapter (shared helper) { display-mode: "form" }
# Shared optimized adapter for all six image benchmarks.
from contextlib import nullcontext
from time import perf_counter

import numpy as np
import torch
from sklearn.model_selection import train_test_split
import mlestar.adapters.vision as vision
import mlestar.experiment as experiment_module
from mlestar.experiment import compare
from mlestar.metrics import score_metric

class FastVisionMixin:
    def __init__(
        self,
        *args,
        num_workers=4,
        use_amp=True,
        min_epochs=3,
        early_stopping_patience=5,
        inner_validation_fraction=0.10,
        max_inner_validation_samples=500,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.num_workers = int(num_workers)
        self.use_amp = bool(use_amp and vision._DEVICE.type == "cuda")
        self.min_epochs = int(min_epochs)
        self.early_stopping_patience = int(early_stopping_patience)
        self.inner_validation_fraction = float(inner_validation_fraction)
        self.max_inner_validation_samples = int(max_inner_validation_samples)
        self._fit_number = 0
        self._epoch_records = []
        self._epoch_context = {}
        if not 1 <= self.min_epochs <= self.epochs:
            raise ValueError("min_epochs must be in [1, max_epochs]")
        if self.early_stopping_patience < 1:
            raise ValueError("early_stopping_patience must be positive")
        if not 0 < self.inner_validation_fraction < 0.5:
            raise ValueError("inner_validation_fraction must be in (0, 0.5)")

    def _loader(self, dataset, *, shuffle, seed=None, drop_last=False):
        options = dict(
            dataset=dataset,
            batch_size=self.batch_size,
            shuffle=shuffle,
            num_workers=self.num_workers,
            pin_memory=(vision._DEVICE.type == "cuda"),
            persistent_workers=(self.num_workers > 0),
            drop_last=bool(drop_last),
        )
        if self.num_workers > 0:
            options["prefetch_factor"] = 2
        if shuffle:
            options["generator"] = torch.Generator().manual_seed(int(seed))
        return torch.utils.data.DataLoader(**options)

    def _scaler(self):
        try:
            return torch.amp.GradScaler("cuda", enabled=self.use_amp)
        except (AttributeError, TypeError):
            return torch.cuda.amp.GradScaler(enabled=self.use_amp)

    def _autocast(self):
        if not self.use_amp:
            return nullcontext()
        return torch.autocast(device_type="cuda", dtype=torch.float16)

    def _fit(self, model, dataset, seed):
        parameters = list(model.parameters())
        if not parameters:
            return
        self._fit_number += 1
        if len(dataset) < 2:
            raise ValueError("Epoch selection requires at least two training rows")
        validation_size = min(
            max(1, int(round(len(dataset) * self.inner_validation_fraction))),
            self.max_inner_validation_samples,
        )
        validation_size = min(validation_size, len(dataset) - 1)
        all_indices = np.arange(len(dataset))
        if self.task.modality == "image_ordinal":
            ordinal_labels = (
                dataset.targets.detach().cpu().numpy().reshape(-1).astype(int)
            )
            fitting_indices, validation_indices = train_test_split(
                all_indices,
                test_size=validation_size,
                random_state=int(seed),
                stratify=ordinal_labels,
            )
        else:
            permutation = torch.randperm(
                len(dataset), generator=torch.Generator().manual_seed(int(seed))
            ).numpy()
            validation_indices = permutation[:validation_size]
            fitting_indices = permutation[validation_size:]
        validation_dataset = torch.utils.data.Subset(
            dataset, validation_indices.tolist()
        )
        fitting_dataset = torch.utils.data.Subset(
            dataset, fitting_indices.tolist()
        )
        initial_state = {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        }
        loader = self._loader(
            fitting_dataset,
            shuffle=True,
            seed=seed,
            drop_last=(
                len(fitting_dataset) > self.batch_size
                and len(fitting_dataset) % self.batch_size == 1
            ),
        )
        validation_loader = self._loader(validation_dataset, shuffle=False)
        optimizer = torch.optim.Adam(parameters, lr=1e-4)
        loss_fn = self._loss_fn()
        scaler = self._scaler()
        metric_name = self.task.metric.name
        greater_is_better = bool(self.task.metric.greater_is_better)
        best_metric = float("-inf") if greater_is_better else float("inf")
        selected_epoch_val_loss = None
        best_epoch = None
        stale_epochs = 0
        for epoch in range(self.epochs):
            started = perf_counter()
            loss_sum = 0.0
            model.train()
            for images, targets in loader:
                images = images.to(vision._DEVICE, non_blocking=True)
                targets = targets.to(vision._DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with self._autocast():
                    loss = loss_fn(model(images), targets)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Non-finite loss: {loss.item()}")
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                loss_sum += float(loss.detach().cpu())

            validation_loss_sum = 0.0
            validation_rows = 0
            validation_predictions = []
            validation_targets = []
            model.eval()
            with torch.inference_mode():
                for images, targets in validation_loader:
                    images = images.to(vision._DEVICE, non_blocking=True)
                    targets = targets.to(vision._DEVICE, non_blocking=True)
                    with self._autocast():
                        logits = model(images)
                        validation_loss = loss_fn(logits, targets)
                    batch_prediction = self._predict_probs(logits)
                    if torch.is_tensor(batch_prediction):
                        batch_prediction = batch_prediction.detach().float().cpu().numpy()
                    validation_predictions.append(np.asarray(batch_prediction))
                    validation_targets.append(
                        targets.detach().float().cpu().numpy()
                    )
                    batch_rows = int(images.shape[0])
                    validation_loss_sum += float(validation_loss.detach().cpu()) * batch_rows
                    validation_rows += batch_rows
            mean_validation_loss = validation_loss_sum / max(validation_rows, 1)
            metric_truth = np.concatenate(validation_targets)
            metric_prediction = np.concatenate(validation_predictions)
            if self.task.modality in {
                "image_binary", "image_ordinal", "image_multiclass"
            }:
                metric_truth = metric_truth.reshape(-1)
            scoring_truth, scoring_prediction = self._score_inputs(
                metric_truth, metric_prediction
            )
            metric_options = {}
            if self.task.modality == "image_multiclass":
                metric_options["labels"] = list(range(metric_prediction.shape[1]))
            current_metric = float(score_metric(
                self.task.metric,
                scoring_truth,
                scoring_prediction,
                **metric_options,
            ).value)
            if not np.isfinite(current_metric):
                improved = False
            elif greater_is_better:
                improved = current_metric > best_metric + 1e-12
            else:
                improved = current_metric < best_metric - 1e-12
            if improved:
                best_metric = current_metric
                selected_epoch_val_loss = mean_validation_loss
                best_epoch = epoch + 1
                stale_epochs = 0
            else:
                stale_epochs += 1
            selection_text = (
                f"inner_val_{metric_name}={current_metric:.5f} "
                f"best_{metric_name}={best_metric:.5f}"
            )
            print(
                f"  fit={self._fit_number} epoch={epoch + 1}/{self.epochs} "
                f"loss={loss_sum / max(len(loader), 1):.5f} "
                f"inner_val_loss={mean_validation_loss:.5f} "
                f"{selection_text} "
                f"best_epoch={best_epoch} "
                f"seconds={perf_counter() - started:.1f}"
            )
            if (
                epoch + 1 >= self.min_epochs
                and stale_epochs >= self.early_stopping_patience
            ):
                break
        if best_epoch is None:
            raise RuntimeError("Epoch selection did not produce a finite checkpoint")
        # The inner split chooses only the epoch count. Rewind the model and
        # train from the same initialization on every available training row,
        # leaving the outer OOF fold untouched.
        model.load_state_dict(initial_state)
        model.to(vision._DEVICE)
        torch.manual_seed(int(seed))
        if vision._DEVICE.type == "cuda":
            torch.cuda.manual_seed_all(int(seed))
        full_loader = self._loader(
            dataset,
            shuffle=True,
            seed=seed,
            drop_last=(len(dataset) > self.batch_size and len(dataset) % self.batch_size == 1),
        )
        full_optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
        full_scaler = self._scaler()
        for refit_epoch in range(best_epoch):
            model.train()
            full_loss_sum = 0.0
            for images, targets in full_loader:
                images = images.to(vision._DEVICE, non_blocking=True)
                targets = targets.to(vision._DEVICE, non_blocking=True)
                full_optimizer.zero_grad(set_to_none=True)
                with self._autocast():
                    full_loss = loss_fn(model(images), targets)
                if not torch.isfinite(full_loss):
                    raise FloatingPointError(f"Non-finite refit loss: {full_loss.item()}")
                full_scaler.scale(full_loss).backward()
                full_scaler.step(full_optimizer)
                full_scaler.update()
                full_loss_sum += float(full_loss.detach().cpu())
            print(
                f"  fit={self._fit_number} full_refit_epoch={refit_epoch + 1}/{best_epoch} "
                f"loss={full_loss_sum / max(len(full_loader), 1):.5f}"
            )
        record = {
            **self._epoch_context,
            "fit_number": self._fit_number,
            "seed": int(seed),
            "dataset_rows": len(dataset),
            "inner_fit_rows": len(fitting_dataset),
            "inner_validation_rows": len(validation_dataset),
            "selected_epoch": int(best_epoch),
            "best_inner_validation_loss": float(selected_epoch_val_loss),
            "best_inner_validation_metric": float(best_metric),
            "best_inner_validation_metric_name": metric_name,
            "metric_greater_is_better": greater_is_better,
            "best_inner_validation_qwk": (
                float(best_metric) if metric_name == "qwk" else None
            ),
            "max_epochs": int(self.epochs),
            "stopped_early": bool(best_epoch < self.epochs),
            "criterion": (
                f"{'maximum' if greater_is_better else 'minimum'} "
                f"inner-validation {metric_name}"
            ),
            "refit_on_all_training_rows": True,
        }
        self._epoch_records.append(record)
        self.artifacts.write_json(
            "epoch_selection.json",
            {"leakage_rule": "outer OOF rows are not used for epoch selection", "records": self._epoch_records},
        )

    def _predict(self, model, dataset):
        loader = self._loader(dataset, shuffle=False)
        outputs = []
        model.eval()
        with torch.inference_mode():
            for images, _ in loader:
                images = images.to(vision._DEVICE, non_blocking=True)
                with self._autocast():
                    outputs.append(self._predict_probs(model(images)))
        return np.concatenate(outputs, axis=0)

    def run(self, candidate, *, phase, seed, parent_experiment_id=None):
        model_name = candidate.block("model")
        print(f"START seed={seed} phase={phase} model={model_name}")
        started = perf_counter()
        self._epoch_context = {
            "phase": phase,
            "candidate_id": candidate.candidate_id,
            "model": model_name,
        }
        result = super().run(
            candidate, phase=phase, seed=seed,
            parent_experiment_id=parent_experiment_id,
        )
        print(
            f"END   seed={seed} phase={phase} model={model_name} "
            f"metric={result.receipt.metric_value} error={result.receipt.error} "
            f"minutes={(perf_counter() - started) / 60:.2f}"
        )
        self._epoch_context = {}
        return result

for benchmark, base_class in list(experiment_module.ADAPTER_CLASSES.items()):
    if issubclass(base_class, vision.ImageClassificationAdapter):
        fast_class = type(f"Fast{base_class.__name__}", (FastVisionMixin, base_class), {})
        experiment_module.ADAPTER_CLASSES[benchmark] = fast_class

image_requested = any(name != "leaf_classification" for name in BENCHMARKS_TO_RUN)
if image_requested and vision._DEVICE.type != "cuda":
    raise RuntimeError("Enable a Colab GPU runtime before running image benchmarks.")
print(
    "device:",
    torch.cuda.get_device_name(0) if vision._DEVICE.type == "cuda" else "CPU (leaf only)",
)


device: NVIDIA L4


In [6]:
#@title Benchmark runner (shared helper) { display-mode: "form" }
# Unified task registry and runner. Each original benchmark cell below calls
# this same function, so paths, seeds and adapter options cannot drift.
import json
import pandas as pd
from benchmarks.catalog import get_task

TASKS = {
    "leaf_classification": {
        "slug": "leaf-classification", "marker": "train.csv",
        "data_root": "/content/leaf-classification", "vision": False,
    },
    "plant_pathology_2020": {
        "slug": "plant-pathology-2020-fgvc7", "marker": "train.csv",
        "data_root": "/content/plant-pathology-2020-fgvc7", "vision": True,
    },
    "aptos_2019": {
        "slug": "aptos2019-blindness-detection", "marker": "train.csv",
        "data_root": "/content/aptos2019-blindness-detection", "vision": True,
    },
    "dog_breed": {
        "slug": "dog-breed-identification", "marker": "labels.csv",
        "data_root": "/content/dog-breed-identification", "vision": True,
    },
    "aerial_cactus": {
        "slug": "aerial-cactus-identification", "marker": "train.csv",
        "data_root": "/content/aerial-cactus-identification", "vision": True,
    },
    "dogs_vs_cats": {
        "slug": "dogs-vs-cats-redux-kernels-edition", "marker": "train",
        "data_root": "/content/dogs-vs-cats-redux-kernels-edition", "vision": True,
    },
    "histopathologic_cancer": {
        "slug": "histopathologic-cancer-detection", "marker": "train_labels.csv",
        "data_root": "/content/histopathologic-cancer-detection", "vision": True,
    },
}
COMPLETED = {}

def run_benchmark(benchmark):
    if benchmark not in BENCHMARKS_TO_RUN:
        print(f"SKIP {benchmark}; add it to BENCHMARKS_TO_RUN to execute.")
        return None
    config = TASKS[benchmark]
    source_root = fetch_kaggle_competition(
        config["slug"], config["data_root"], config["marker"]
    )
    data_root = prepare_aptos_cache(source_root) if benchmark == "aptos_2019" else source_root
    run_root = RUNS_ROOT / benchmark
    comparison_path = run_root / "comparison.csv"
    receipts_path = run_root / "receipts.jsonl"
    report_path = run_root / "comparison.json"
    if RESUME_EXISTING_RUNS and comparison_path.is_file() and receipts_path.is_file():
        comparison = pd.read_csv(comparison_path)
        expected_rows = {(seed, arm) for seed in SEEDS for arm in (
            "baseline", "mlestar_initial", "mlestar_refined", "mlestar_ensemble"
        )}
        actual_rows = set(zip(comparison["seed"].astype(int), comparison["arm"].astype(str)))
        if actual_rows != expected_rows:
            raise RuntimeError(
                f"Incomplete or incompatible cached comparison for {benchmark}; "
                f"expected {len(expected_rows)} seed/arm rows, got {len(actual_rows)}"
            )
        epoch_evidence_paths = list(run_root.rglob("epoch_selection.json"))
        has_epoch_evidence = bool(epoch_evidence_paths)
        if (
            config["vision"] and not has_epoch_evidence
            and not ALLOW_LEGACY_FIXED_EPOCH_OOF_RESUME
        ):
            raise RuntimeError(
                f"{benchmark} has legacy fixed-epoch OOF artifacts. Set "
                "ALLOW_LEGACY_FIXED_EPOCH_OOF_RESUME=True to reuse them, or delete "
                "the run directory to recompute with MLE-selected epochs."
            )
        if not config["vision"]:
            protocol = "not_applicable_tree_models"
        elif has_epoch_evidence:
            epoch_records = [
                record
                for path in epoch_evidence_paths
                for record in json.loads(path.read_text()).get("records", [])
            ]
            metric = get_task(benchmark).metric
            expected_criterion = (
                f"{'maximum' if metric.greater_is_better else 'minimum'} "
                f"inner-validation {metric.name}"
            )
            if not epoch_records or any(
                record.get("criterion") != expected_criterion
                for record in epoch_records
            ):
                raise RuntimeError(
                    f"Cached {benchmark} artifacts did not select epochs by the "
                    f"official metric ({expected_criterion}). Move or delete that "
                    "run directory before the formal rerun."
                )
            protocol = "mle_inner_official_metric_selected_epochs"
        else:
            protocol = "legacy_fixed_3_epochs"
        report = (
            json.loads(report_path.read_text())
            if report_path.is_file()
            else {"benchmark": benchmark, "status": "resumed_from_artifacts"}
        )
        COMPLETED[benchmark] = {
            "data_root": str(data_root), "run_root": str(run_root),
            "report": report, "resumed": True, "oof_epoch_protocol": protocol,
        }
        print(f"RESUME {benchmark}: OOF comparison skipped ({protocol}).")
        display(comparison)
        display(comparison.groupby("arm")["metric_value"].agg(["mean", "std", "count"]))
        return report
    adapter_kwargs = {}
    if config["vision"]:
        adapter_kwargs = {
            "pretrained": True,
            "image_size": IMAGE_SIZE,
            "epochs": MAX_EPOCHS,
            "batch_size": BATCH_SIZE,
            "max_train_samples": MAX_TRAIN_SAMPLES_PER_FOLD,
            "num_workers": NUM_WORKERS,
            "use_amp": USE_AMP,
            "min_epochs": MIN_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "inner_validation_fraction": INNER_VALIDATION_FRACTION,
        }
    report = compare(
        benchmark=benchmark,
        data_root=data_root,
        run_root=run_root,
        seeds=SEEDS,
        outer_rounds=1,
        inner_rounds=1,
        adapter_kwargs=adapter_kwargs,
    )
    comparison = pd.read_csv(run_root / "comparison.csv")
    COMPLETED[benchmark] = {
        "data_root": str(data_root), "run_root": str(run_root), "report": report,
        "resumed": False,
        "oof_epoch_protocol": (
            "mle_inner_official_metric_selected_epochs"
            if config["vision"] else "not_applicable_tree_models"
        ),
    }
    display(comparison)
    display(comparison.groupby("arm")["metric_value"].agg(["mean", "std", "count"]))
    return report


In [8]:
#@title Kaggle prediction and submission (shared helper) { display-mode: "form" }
# Generic final prediction, validation, Kaggle API submission and status capture.
# Each dataset cell below calls run_complete_benchmark exactly once.
import json
import os
import subprocess
from io import StringIO
from pathlib import Path
from time import monotonic, sleep

import numpy as np
import pandas as pd
from timm import create_model
from benchmarks.catalog import get_task
from mlestar.ensemble import blend_test_predictions, select_ensemble

PIPELINE_RESULTS = {}

def build_mle_ensemble_plans(benchmark, data_root, run_root):
    task = get_task(benchmark)
    receipts = [
        json.loads(line)
        for line in (Path(run_root) / "receipts.jsonl").read_text().splitlines()
        if line.strip()
    ]
    if benchmark == "leaf_classification":
        labels = pd.read_csv(Path(data_root) / "train.csv")["species"].astype(str).to_numpy()
    else:
        _, _, labels, _ = vision_data(benchmark, data_root)
    plans = []
    for seed in SEEDS:
        def receipt_for(phase):
            matched = [
                row for row in receipts
                if row.get("phase") == phase and int(row.get("seed")) == seed
                and row.get("metric_value") is not None
            ]
            if len(matched) != 1:
                raise RuntimeError(
                    f"Expected one successful {phase} receipt for {benchmark}, seed {seed}"
                )
            return matched[0]

        baseline = receipt_for("baseline")
        initial_selected = receipt_for("initial_selected")
        refined = receipt_for("refined_selected")
        baseline_oof = pd.read_csv(Path(run_root) / f"seed_{seed}" / baseline["oof_path"])
        refined_oof = pd.read_csv(Path(run_root) / f"seed_{seed}" / refined["oof_path"])
        baseline_values = baseline_oof.iloc[:, 1:].to_numpy(dtype=float)
        refined_values = refined_oof.iloc[:, 1:].to_numpy(dtype=float)
        if baseline_values.shape[1] == 1:
            baseline_values = baseline_values[:, 0]
            refined_values = refined_values[:, 0]
        score_transform = None
        if task.modality == "image_ordinal":
            max_label = int(np.max(labels))
            score_transform = lambda prediction, bound=max_label: np.clip(
                np.rint(prediction), 0, bound
            )
        ensemble = select_ensemble(
            {
                "baseline": (baseline_oof.iloc[:, 0].astype(str).tolist(), baseline_values),
                "refined": (refined_oof.iloc[:, 0].astype(str).tolist(), refined_values),
            },
            labels,
            task.metric,
            score_transform=score_transform,
        )
        comparison = pd.read_csv(Path(run_root) / "comparison.csv")
        reported = comparison.loc[
            (comparison["seed"] == seed)
            & (comparison["arm"] == "mlestar_ensemble"),
            "metric_value",
        ]
        if len(reported) != 1 or not np.isclose(
            float(reported.iloc[0]), float(ensemble.score.value), rtol=0, atol=1e-12
        ):
            raise AssertionError(
                f"Reconstructed ensemble score does not match comparison.csv for seed {seed}"
            )
        plan = {
            "seed": seed,
            "weights": ensemble.weights,
            "oof_score": ensemble.score.value,
            "baseline_receipt": baseline,
            "refined_receipt": refined,
        }
        if benchmark != "leaf_classification":
            initial_id = str(initial_selected["candidate_id"])
            refined_id = str(refined["candidate_id"])
            if refined_id == initial_id:
                refined_model = initial_id
            elif refined_id == f"{initial_id}-model":
                refined_model = (
                    "efficientnet_b0" if initial_id == "resnet18" else "resnet18"
                )
            else:
                raise RuntimeError(
                    f"Cannot reconstruct refined model from initial={initial_id}, refined={refined_id}"
                )
            if refined_model not in {"resnet18", "efficientnet_b0"}:
                raise RuntimeError(f"Unexpected reconstructed model: {refined_model}")
            plan["baseline_model"] = "resnet18"
            plan["refined_model"] = refined_model
        plans.append(plan)
    display(pd.DataFrame([
        {
            "seed": plan["seed"],
            "baseline_weight": plan["weights"]["baseline"],
            "refined_weight": plan["weights"]["refined"],
            "refined_model": plan.get("refined_model", "receipt test predictions"),
            "reconstructed_oof_score": plan["oof_score"],
        }
        for plan in plans
    ]))
    return plans

def build_leaf_submission(data_root, run_root, plans):
    sample = pd.read_csv(Path(data_root) / "sample_submission.csv")
    id_column = sample.columns[0]
    prediction_frames = []
    for plan in plans:
        seed = plan["seed"]
        candidates = {}
        for name, receipt in (
            ("baseline", plan["baseline_receipt"]),
            ("refined", plan["refined_receipt"]),
        ):
            if not receipt.get("test_path"):
                raise RuntimeError(f"Leaf {name} receipt has no test predictions for seed {seed}")
            frame = pd.read_csv(Path(run_root) / f"seed_{seed}" / receipt["test_path"])
            if list(frame.columns) != list(sample.columns):
                raise ValueError("Leaf prediction columns differ from sample_submission.csv")
            if frame[id_column].astype(str).tolist() != sample[id_column].astype(str).tolist():
                raise ValueError("Leaf test IDs or order differ from sample_submission.csv")
            candidates[name] = frame.iloc[:, 1:].to_numpy(dtype=float)
        prediction_frames.append(blend_test_predictions(candidates, plan["weights"]))
    submission = pd.DataFrame(
        np.mean(np.stack(prediction_frames), axis=0),
        columns=list(sample.columns[1:]),
    )
    submission.insert(0, id_column, sample[id_column].to_numpy())
    return submission

def vision_data(benchmark, data_root):
    root = Path(data_root)
    sample = pd.read_csv(root / "sample_submission.csv")
    if benchmark == "plant_pathology_2020":
        train = pd.read_csv(root / "train.csv")
        labels = train[["healthy", "multiple_diseases", "rust", "scab"]].to_numpy(float)
        train_paths = [root / "images" / f"{value}.jpg" for value in train["image_id"].astype(str)]
        test_paths = [root / "images" / f"{value}.jpg" for value in sample["image_id"].astype(str)]
    elif benchmark == "aptos_2019":
        train = pd.read_csv(root / "train.csv")
        labels = train["diagnosis"].to_numpy(float)
        train_paths = [root / "train_images" / f"{value}.png" for value in train["id_code"].astype(str)]
        test_paths = [root / "test_images" / f"{value}.png" for value in sample["id_code"].astype(str)]
    elif benchmark == "dog_breed":
        train = pd.read_csv(root / "labels.csv")
        classes = list(sample.columns[1:])
        mapping = {name: index for index, name in enumerate(classes)}
        if train["breed"].map(mapping).isna().any():
            raise ValueError("Dog Breed labels contain a class absent from sample_submission.csv")
        labels = train["breed"].map(mapping).to_numpy(int)
        train_paths = [root / "train" / f"{value}.jpg" for value in train["id"].astype(str)]
        test_paths = [root / "test" / f"{value}.jpg" for value in sample["id"].astype(str)]
    elif benchmark == "aerial_cactus":
        train = pd.read_csv(root / "train.csv")
        labels = train["has_cactus"].to_numpy(float)
        train_paths = [root / "train" / value for value in train["id"].astype(str)]
        test_paths = [root / "test" / value for value in sample["id"].astype(str)]
    elif benchmark == "dogs_vs_cats":
        train_paths = sorted((root / "train").glob("*.jpg"))
        labels = np.asarray([1.0 if path.name.startswith("dog.") else 0.0 for path in train_paths])
        test_paths = [root / "test" / f"{value}.jpg" for value in sample["id"].astype(str)]
    elif benchmark == "histopathologic_cancer":
        train = pd.read_csv(root / "train_labels.csv")
        labels = train["label"].to_numpy(float)
        train_paths = [root / "train" / f"{value}.tif" for value in train["id"].astype(str)]
        test_paths = [root / "test" / f"{value}.tif" for value in sample["id"].astype(str)]
    else:
        raise KeyError(benchmark)
    missing = [str(path) for path in train_paths + test_paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing {benchmark} images, examples: {missing[:5]}")
    return sample, train_paths, labels, test_paths

def build_vision_submission(benchmark, data_root, run_root, plans):
    task = get_task(benchmark)
    sample, train_paths, labels, test_paths = vision_data(benchmark, data_root)
    adapter_class = experiment_module.ADAPTER_CLASSES[benchmark]
    predictions = []
    checkpoint_root = Path(run_root) / "final_checkpoints"
    checkpoint_root.mkdir(parents=True, exist_ok=True)
    for plan in plans:
        seed = plan["seed"]
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        adapter = adapter_class(
            data_root, Path(run_root) / f"final_seed_{seed}", task,
            pretrained=True, image_size=IMAGE_SIZE, epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE, max_train_samples=None,
            num_workers=NUM_WORKERS, use_amp=USE_AMP,
            min_epochs=MIN_EPOCHS,
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            inner_validation_fraction=INNER_VALIDATION_FRACTION,
        )
        targets = adapter._prepare_targets(np.asarray(labels))
        train_dataset = vision._ImageDataset(train_paths, targets, IMAGE_SIZE)
        if task.modality == "image_multiclass":
            dummy = torch.zeros(len(test_paths), dtype=torch.long)
        elif task.modality == "image_multilabel":
            dummy = torch.zeros((len(test_paths), len(task.target_columns)), dtype=torch.float32)
        else:
            dummy = torch.zeros((len(test_paths), 1), dtype=torch.float32)
        test_dataset = vision._ImageDataset(test_paths, dummy, IMAGE_SIZE)
        num_classes = adapter._num_classes(np.asarray(labels))
        candidate_predictions = {}
        weighted_models = {}
        for arm_name in ("baseline", "refined"):
            weight = float(plan["weights"][arm_name])
            if weight > 0.0:
                model_name = plan[f"{arm_name}_model"]
                weighted_models[model_name] = weighted_models.get(model_name, 0.0) + weight
        if not weighted_models:
            raise RuntimeError(f"Seed {seed} has no positive MLE ensemble weight")
        for model_name in sorted(weighted_models):
            # Seed each candidate independently, matching the adapter's OOF
            # model construction instead of letting one model consume the
            # next model's random-number stream.
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            model = create_model(model_name, pretrained=True, num_classes=num_classes).to(vision._DEVICE)
            checkpoint_path = checkpoint_root / f"{model_name}_seed_{seed}.pt"
            resumed = False
            if RESUME_EXISTING_RUNS and checkpoint_path.is_file():
                try:
                    checkpoint = torch.load(
                        checkpoint_path, map_location=vision._DEVICE, weights_only=False
                    )
                except TypeError:  # compatibility with older PyTorch
                    checkpoint = torch.load(checkpoint_path, map_location=vision._DEVICE)
                if checkpoint.get("model_name") != model_name or int(checkpoint.get("seed")) != int(seed):
                    raise ValueError(f"Checkpoint metadata mismatch: {checkpoint_path}")
                model.load_state_dict(checkpoint["model"])
                resumed = True
                print(f"Resume final checkpoint: benchmark={benchmark} seed={seed} model={model_name}")
            if not resumed:
                print(f"Final fit: benchmark={benchmark} seed={seed} model={model_name}")
                adapter._epoch_context = {
                    "phase": "final_full_data",
                    "candidate_id": model_name,
                    "model": model_name,
                }
                adapter._fit(model, train_dataset, seed)
                temporary_path = checkpoint_path.with_suffix(".tmp.pt")
                torch.save(
                    {"model": model.state_dict(), "model_name": model_name, "seed": seed},
                    temporary_path,
                )
                temporary_path.replace(checkpoint_path)
            candidate_predictions[model_name] = np.asarray(
                adapter._predict(model, test_dataset), dtype=np.float64
            )
            del model
            torch.cuda.empty_cache()
        predictions.append(sum(
            weight * candidate_predictions[model_name]
            for model_name, weight in weighted_models.items()
        ))
        torch.cuda.empty_cache()
    raw = np.stack(predictions, axis=0)
    mean_prediction = raw.mean(axis=0)
    submission = sample.copy()
    if task.modality == "image_ordinal":
        submission[task.submission.prediction_columns[0]] = np.clip(
            np.rint(mean_prediction), 0, int(np.max(labels))
        ).astype(int)
    elif mean_prediction.ndim == 1:
        submission[task.submission.prediction_columns[0]] = mean_prediction
    else:
        columns = list(sample.columns[1:]) if task.submission.prediction_from_sample else list(task.target_columns)
        if mean_prediction.shape != (len(sample), len(columns)):
            raise ValueError(
                f"Prediction shape {mean_prediction.shape} does not match {len(columns)} columns"
            )
        submission[columns] = mean_prediction
    np.save(Path(run_root) / "raw_test_predictions.npy", raw)
    return submission

def validate_submission(benchmark, data_root, submission):
    task = get_task(benchmark)
    sample = pd.read_csv(Path(data_root) / "sample_submission.csv")
    id_column = task.submission.id_columns[0]
    if list(submission.columns) != list(sample.columns):
        raise AssertionError(f"{benchmark}: columns differ from sample_submission.csv")
    if len(submission) != len(sample):
        raise AssertionError(f"{benchmark}: row count differs from sample_submission.csv")
    if submission[id_column].astype(str).tolist() != sample[id_column].astype(str).tolist():
        raise AssertionError(f"{benchmark}: IDs or order differ from sample_submission.csv")
    values = submission.drop(columns=[id_column]).to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise AssertionError(f"{benchmark}: predictions contain NaN or infinity")
    if task.modality == "image_ordinal":
        if not np.equal(values, np.rint(values)).all() or not ((values >= 0) & (values <= 4)).all():
            raise AssertionError("APTOS predictions must be integer grades 0..4")
    elif not ((values >= 0) & (values <= 1)).all():
        raise AssertionError(f"{benchmark}: probabilities are outside [0, 1]")
    print(f"Validated {benchmark} submission: rows={len(submission)}, columns={len(submission.columns)}")

def collect_epoch_selection(run_root):
    records = []
    for path in sorted(Path(run_root).rglob("epoch_selection.json")):
        payload = json.loads(path.read_text())
        for record in payload.get("records", []):
            records.append({"artifact": str(path.relative_to(run_root)), **record})
    if not records:
        return [], None
    frame = pd.DataFrame(records)
    output_path = Path(run_root) / "selected_epochs.csv"
    frame.to_csv(output_path, index=False)
    display(frame)
    display(
        frame.groupby(["phase", "model"])["selected_epoch"]
        .agg(["median", "mean", "min", "max", "count"])
    )
    print("Epoch-selection evidence:", output_path)
    return records, str(output_path)

def kaggle_submit_and_capture(benchmark, submission_path):
    slug = TASKS[benchmark]["slug"]
    message = f"{SUBMISSION_PREFIX} | {benchmark}"
    record = {"requested": SUBMIT_ALL_VIA_API, "accepted": False, "message": message}
    if not SUBMIT_ALL_VIA_API:
        record["status"] = "disabled"
        return record
    if not os.environ.get("KAGGLE_API_TOKEN"):
        record.update(status="error", error="KAGGLE_API_TOKEN is not configured")
        return record
    completed = subprocess.run(
        ["kaggle", "competitions", "submit", slug, "-f", str(submission_path), "-m", message],
        text=True, capture_output=True,
    )
    record.update(
        accepted=(completed.returncode == 0),
        submit_stdout=completed.stdout.strip(),
        submit_stderr=completed.stderr.strip(),
        submit_returncode=completed.returncode,
    )
    print(completed.stdout)
    print(completed.stderr)
    if completed.returncode != 0:
        record["status"] = "api_rejected"
        return record

    deadline = monotonic() + KAGGLE_SCORE_POLL_SECONDS
    while True:
        listing = subprocess.run(
            ["kaggle", "competitions", "submissions", slug, "-v"],
            text=True, capture_output=True,
        )
        record["submissions_stdout"] = listing.stdout.strip()
        record["submissions_stderr"] = listing.stderr.strip()
        try:
            # Older Kaggle CLI versions print an upgrade warning before
            # the CSV header. Start parsing at the actual header so that
            # publicScore/privateScore cannot shift into bogus keys.
            output_lines = listing.stdout.splitlines()
            header_index = next(
                index for index, line in enumerate(output_lines)
                if line.lstrip().startswith("fileName,")
            )
            table = pd.read_csv(StringIO("\n".join(output_lines[header_index:])))
            description_column = next(
                (name for name in table.columns if name.lower() == "description"), None
            )
            matched = table[table[description_column] == message] if description_column else table.head(1)
            if not matched.empty:
                latest = matched.iloc[0].replace({np.nan: None}).to_dict()
                record["latest_submission"] = latest
                status = str(latest.get("status", latest.get("Status", ""))).lower()
                if status and status not in {"pending", "running"}:
                    break
        except Exception as error:
            record["listing_parse_error"] = f"{type(error).__name__}: {error}"
        if monotonic() >= deadline:
            break
        sleep(15)
    record["status"] = "accepted"
    return record

def run_complete_benchmark(benchmark):
    result = {"benchmark": benchmark, "stage": "starting", "success": False}
    PIPELINE_RESULTS[benchmark] = result
    if benchmark not in BENCHMARKS_TO_RUN:
        result.update(stage="skipped", error=None)
        print(f"SKIP {benchmark}")
        return result
    try:
        result["stage"] = "data_download_and_oof_comparison"
        run_benchmark(benchmark)
        info = COMPLETED[benchmark]
        data_root = Path(info["data_root"])
        run_root = Path(info["run_root"])
        plans = build_mle_ensemble_plans(benchmark, data_root, run_root)
        result["oof_epoch_protocol"] = info.get("oof_epoch_protocol")
        result["resumed_oof"] = bool(info.get("resumed"))
        result["final_protocol"] = (
            "per-seed mlestar_ensemble OOF weights, then 3-seed mean; "
            + str(info.get("oof_epoch_protocol", "unknown epoch protocol"))
        )
        result["ensemble_plans"] = plans

        result["stage"] = "final_prediction"
        if benchmark == "leaf_classification":
            submission = build_leaf_submission(data_root, run_root, plans)
        else:
            submission = build_vision_submission(benchmark, data_root, run_root, plans)
        epoch_records, epoch_path = collect_epoch_selection(run_root)
        result["epoch_selection"] = {
            "method": (
                "not applicable to tree models"
                if benchmark == "leaf_classification"
                else (
                    f"{'maximum' if get_task(benchmark).metric.greater_is_better else 'minimum'} "
                    f"{get_task(benchmark).metric.name} on an inner split of each "
                    "training fold"
                )
            ),
            "evidence_path": epoch_path,
            "records": epoch_records,
        }
        validate_submission(benchmark, data_root, submission)
        submission_path = run_root / "submission.csv"
        submission.to_csv(submission_path, index=False)
        result["submission_path"] = str(submission_path)
        display(submission.head())

        result["stage"] = "kaggle_api"
        result["kaggle"] = kaggle_submit_and_capture(benchmark, submission_path)
        result.update(stage="complete", success=True)
    except Exception as error:
        result.update(
            success=False,
            error=f"{type(error).__name__}: {error}",
            failed_stage=result["stage"],
            stage="failed",
        )
        print(f"FAILED {benchmark}: {result['error']}")
        if STOP_ON_PIPELINE_FAILURE:
            raise
    finally:
        result_path = RUNS_ROOT / f"{benchmark}_pipeline_result.json"
        result_path.write_text(json.dumps(result, indent=2, default=str), encoding="utf-8")
        print(json.dumps(result, indent=2, default=str))
    return result


In [10]:
# Dog Breed Identification.
DATA_ROOT = "/content/dog-breed-identification"
RUN_ROOT = RUNS_ROOT / "dog_breed"

# OOF comparison -> MLE ensemble reconstruction -> test prediction ->
# sample_submission validation -> Kaggle API submission.
result = run_complete_benchmark("dog_breed")
pd.read_csv(RUN_ROOT / "comparison.csv") if (RUN_ROOT / "comparison.csv").is_file() else result


START seed=13 phase=baseline model=resnet18


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


  fit=1 epoch=1/40 loss=4.79712 inner_val_loss=4.72916 inner_val_multiclass_log_loss=4.72895 best_multiclass_log_loss=4.72895 best_epoch=1 seconds=2.5


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


  fit=1 epoch=2/40 loss=4.63418 inner_val_loss=4.63875 inner_val_multiclass_log_loss=4.63864 best_multiclass_log_loss=4.63864 best_epoch=2 seconds=2.1


...

median       mean  min  max  count
phase            model                                              
baseline         resnet18           38.0  37.000000   29   40     15
final_full_data  resnet18           12.0  11.666667   10   13      3
initial          efficientnet_b0    27.0  25.600000   16   31     15
                 resnet18           38.0  37.000000   29   40     15
initial_selected resnet18           38.0  37.000000   29   40     15
refined_selected resnet18           38.0  37.000000   29   40     15
refinement       efficientnet_b0    27.0  25.600000   16   31     15

Epoch-selection evidence: /content/drive/MyDrive/mlestar-runs/dog_breed/selected_epochs.csv
Validated dog_breed submission: rows=10357, columns=121


,id,affenpinscher,afghan_hound,african_hunting_dog,airedale,american_staffordshire_terrier,appenzeller,australian_terrier,basenji,basset,...,toy_poodle,toy_terrier,vizsla,walker_hound,weimaraner,welsh_springer_spaniel,west_highland_white_terrier,whippet,wire-haired_fox_terrier,yorkshire_terrier
0,000621fb3cbb32d8935728e48679680e,0.000321,0.005648,0.000056,0.000022,0.000143,0.000152,0.000068,8.635768e-05,0.000313,...,0.000180,0.000537,0.000024,0.000149,0.000058,0.000535,0.000037,0.000456,0.000115,0.000522
1,00102ee9d8eb90812350685311fe5890,0.000103,0.000114,0.000062,0.000048,0.000433,0.000119,0.000251,3.191163e-04,0.000026,...,0.000767,0.000390,0.000033,0.000071,0.000163,0.000041,0.018326,0.000125,0.000584,0.000219
2,0012a730dfa437f5f3613fb75efcd4ce,0.000033,0.126507,0.000061,0.000064,0.000200,0.000031,0.000019,1.709111e-05,0.000260,...,0.000204,0.000063,0.000022,0.000449,0.002044,0.002678,0.003558,0.003914,0.001852,0.000083
3,001510bc8570bbeee98c8d80c8a95ec1,0.003607,0.035428,0.001344,0.002456,0.008212,0.004932,0.002095,2.394399e-02,0.005560,...,0.000525,0.001905,0.006635,0.002863,0.047651,0.002256,0.002261,0.049849,0.001953,0.003492
4,001a5f3114548acdefa3d4da05474c2e,0.001742,0.000703,0.000151,0.000012,0.000022,0.000007,0.000349,4.151546e-07,0.000012,...,0.000284,0.000005,0.000003,0.000001,0.000011,0.000004,0.000080,0.000040,0.000039,0.001020


Successfully submitted to Dog Breed Identification

  0%|          | 0.00/26.5M [00:00<?, ?B/s]
  5%|▍         | 1.27M/26.5M [00:00<00:16, 1.60MB/s]
  6%|▌         | 1.64M/26.5M [00:01<00:16, 1.53MB/s]
  8%|▊         | 2.14M/26.5M [00:01<00:15, 1.64MB/s]
 49%|████▉     | 13.0M/26.5M [00:01<00:00, 14.3MB/s]
 61%|██████    | 16.0M/26.5M [00:01<00:00, 14.0MB/s]
 73%|███████▎  | 19.3M/26.5M [00:02<00:00, 15.6MB/s]
 85%|████████▌ | 22.6M/26.5M [00:02<00:00, 18.0MB/s]
 97%|█████████▋| 25.8M/26.5M [00:02<00:00, 18.9MB/s]
100%|██████████| 26.5M/26.5M [00:03<00:00, 7.90MB/s]

{
  "benchmark": "dog_breed",
  "stage": "complete",
  "success": true,
  "oof_epoch_protocol": "mle_inner_official_metric_selected_epochs",
  "resumed_oof": false,
  "final_protocol": "per-seed mlestar_ensemble OOF weights, then 3-seed mean; mle_inner_official_metric_selected_epochs",
  "ensemble_plans": [
    {
      "seed": 13,
      "weights": {
        "baseline": 0.0,
        "refined": 1.0
      },
      "oof_score"

,seed,arm,metric_value
0,13,baseline,2.458749
1,13,mlestar_initial,2.458749
2,13,mlestar_refined,2.458749
3,13,mlestar_ensemble,2.458749
4,29,baseline,2.471292
5,29,mlestar_initial,2.471292
6,29,mlestar_refined,2.471292
7,29,mlestar_ensemble,2.471292
8,47,baseline,2.464116
9,47,mlestar_initial,2.464116
